# Crypto Market Analysis — BTC / ETH / BNB / SOL (2021–2024)
**Multi-asset technical analysis, regime detection, and risk-adjusted performance**

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
colors_map = {"BTC": "#f7931a", "ETH": "#627eea", "BNB": "#f3ba2f", "SOL": "#9945ff"}


## 1. Load & Inspect

In [2]:
df = pd.read_csv("crypto_raw.csv", parse_dates=["Date"])
df = df.sort_values(["ticker", "Date"]).reset_index(drop=True)
print(f"Shape: {df.shape}")
print(f"Tickers: {df['ticker'].unique()}")
print(f"Date range: {df['Date'].min().date()} → {df['Date'].max().date()}")
df.head(8)


Shape: (5844, 7)
Tickers: <StringArray>
['BNB', 'BTC', 'ETH', 'SOL']
Length: 4, dtype: str
Date range: 2021-01-01 → 2024-12-31


,Date,Open,High,Low,Close,Volume,ticker
0,2021-01-01,38.1915,38.8183,37.6197,38.4903,"727,665,769.0000",BNB
1,2021-01-02,38.2282,38.7307,37.3059,38.1824,"1,520,521,992.0000",BNB
2,2021-01-03,37.9898,42.2195,37.4266,42.0811,"1,099,438,117.0000",BNB
3,2021-01-04,42.4305,43.2104,41.9281,42.1568,"1,996,170,223.0000",BNB
4,2021-01-05,42.0433,45.9950,41.6851,45.7498,"3,002,328,317.0000",BNB
5,2021-01-06,46.2027,46.7671,43.0778,43.4091,"367,108,903.0000",BNB
6,2021-01-07,43.3313,43.8994,39.7933,40.4483,"1,355,243,220.0000",BNB
7,2021-01-08,40.6496,41.5432,38.3193,38.4659,"1,767,819,145.0000",BNB


## 2. Data Cleaning

In [3]:
df["Date"] = pd.to_datetime(df["Date"], utc=False)
df = df.dropna(subset=["Close", "Open", "High", "Low", "Volume"])
df = df[df["High"] >= df["Low"]]
df = df[df["Close"] > 0]
df = df.drop_duplicates(subset=["Date", "ticker"])

for col in ["Open", "High", "Low", "Close"]:
    groups = []
    for t in df["ticker"].unique():
        sub = df[df["ticker"] == t].copy()
        q_low  = sub[col].quantile(0.001)
        q_high = sub[col].quantile(0.999)
        sub[col] = sub[col].clip(lower=q_low, upper=q_high)
        groups.append(sub)
    df = pd.concat(groups).sort_values(["ticker", "Date"]).reset_index(drop=True)

df["Volume"] = df["Volume"].clip(lower=1)

print(f"Nulls remaining  : {df.isnull().sum().sum()}")
print(f"Rows after cleaning: {len(df):,}")
df.describe()


Nulls remaining  : 0
Rows after cleaning: 5,844


,Date,Open,High,Low,Close,Volume
count,5844,"5,844.0000","5,844.0000","5,844.0000","5,844.0000","5,844.0000"
mean,2023-01-01 00:00:00,"29,576.2591","30,529.0647","28,741.0195","29,694.5946","11,427,279,855.4288"
min,2021-01-01 00:00:00,1.6513,1.7862,1.6205,1.7081,"295,318,268.0000"
25%,2022-01-01 00:00:00,28.1426,29.2109,27.1579,28.1925,"1,696,468,607.0000"
50%,2023-01-01 00:00:00,181.2536,186.5114,175.6356,180.8997,"5,035,124,104.0000"
75%,2024-01-01 00:00:00,"5,031.2846","5,157.6490","4,921.1657","5,048.1533","17,462,614,897.0000"
max,2024-12-31 00:00:00,"683,193.6291","716,819.1653","668,356.0660","704,702.4494","126,522,772,239.0000"
std,NaN,"84,295.0882","87,081.0702","81,937.9492","84,769.4638","13,702,459,820.5196"


## 3. Feature Engineering

In [4]:
def build_features(sub):
    sub = sub.copy().sort_values("Date")

    sub["daily_return"]  = sub["Close"].pct_change()
    sub["log_return"]    = np.log(sub["Close"] / sub["Close"].shift(1))
    sub["hl_spread_pct"] = (sub["High"] - sub["Low"]) / sub["Low"] * 100

    for w in [7, 14, 30, 90]:
        sub[f"sma_{w}"]   = sub["Close"].rolling(w).mean()
        sub[f"vol_{w}d"]  = sub["daily_return"].rolling(w).std() * np.sqrt(365)

    sub["sma_20"]      = sub["Close"].rolling(20).mean()
    sub["std_20"]      = sub["Close"].rolling(20).std()
    sub["bb_upper"]    = sub["sma_20"] + 2 * sub["std_20"]
    sub["bb_lower"]    = sub["sma_20"] - 2 * sub["std_20"]
    sub["bb_width"]    = (sub["bb_upper"] - sub["bb_lower"]) / sub["sma_20"]
    sub["bb_position"] = (sub["Close"] - sub["bb_lower"]) / (sub["bb_upper"] - sub["bb_lower"])

    delta = sub["Close"].diff()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    sub["rsi"] = 100 - (100 / (1 + gain / loss.replace(0, np.nan)))

    ema12 = sub["Close"].ewm(span=12, adjust=False).mean()
    ema26 = sub["Close"].ewm(span=26, adjust=False).mean()
    sub["macd"]        = ema12 - ema26
    sub["macd_signal"] = sub["macd"].ewm(span=9, adjust=False).mean()
    sub["macd_hist"]   = sub["macd"] - sub["macd_signal"]

    sub["volume_sma_20"] = sub["Volume"].rolling(20).mean()
    sub["volume_ratio"]  = sub["Volume"] / sub["volume_sma_20"]
    sub["vol_spike"]     = sub["volume_ratio"] > 2.0

    sub["year"]    = sub["Date"].dt.year
    sub["month"]   = sub["Date"].dt.month
    sub["quarter"] = sub["Date"].dt.quarter
    sub["dow"]     = sub["Date"].dt.dayofweek

    return sub

results = []
for t in df["ticker"].unique():
    results.append(build_features(df[df["ticker"] == t]))
df = pd.concat(results, ignore_index=True)
df = df.dropna(subset=["sma_30", "rsi", "macd"])

print(f"Engineered features: {df.shape[1]} columns, {len(df):,} rows")
df[["ticker", "Date", "Close", "rsi", "macd", "vol_30d", "bb_position"]].tail(6)


Engineered features: 35 columns, 5,728 rows


,ticker,Date,Close,rsi,macd,vol_30d,bb_position
5838,SOL,2024-12-26,19.2995,48.5648,0.1872,1.3172,0.4806
5839,SOL,2024-12-27,18.6555,51.6738,0.0865,1.2112,0.4082
5840,SOL,2024-12-28,16.6677,47.3999,-0.1519,1.2686,0.1761
5841,SOL,2024-12-29,15.9632,46.5951,-0.3932,1.2117,0.1324
5842,SOL,2024-12-30,15.7208,47.0265,-0.5971,1.1948,0.1452
5843,SOL,2024-12-31,15.2303,43.6206,-0.7891,1.1972,0.1357


## 4. Market Regime Detection (K-Means, 3 States)

In [5]:
def assign_regime(sub):
    sub = sub.copy().sort_values("Date")
    feats = sub[["log_return", "vol_30d", "rsi", "macd_hist", "bb_position", "volume_ratio"]].dropna()
    if len(feats) < 90:
        sub["regime"] = "Sideways"
        return sub

    scaler  = StandardScaler()
    X       = scaler.fit_transform(feats)
    km      = KMeans(n_clusters=3, random_state=42, n_init=10)
    labels  = km.fit_predict(X)

    centers = pd.DataFrame(
        scaler.inverse_transform(km.cluster_centers_),
        columns=feats.columns
    )
    regime_map = {}
    for i, row in centers.iterrows():
        if row["log_return"] > 0.001 and row["rsi"] > 55:
            regime_map[i] = "Bull"
        elif row["log_return"] < -0.001 and row["rsi"] < 45:
            regime_map[i] = "Bear"
        else:
            regime_map[i] = "Sideways"

    regimes        = pd.Series(labels, index=feats.index).map(regime_map)
    sub["regime"]  = regimes
    sub["regime"]  = sub["regime"].fillna("Sideways")
    return sub

regime_results = []
for t in df["ticker"].unique():
    regime_results.append(assign_regime(df[df["ticker"] == t]))
df = pd.concat(regime_results, ignore_index=True)

df.groupby(["ticker", "regime"]).size().unstack(fill_value=0)


regime,Bear,Bull,Sideways
ticker,,,
BNB,471,571,390
BTC,673,758,1
ETH,425,443,564
SOL,420,437,575


## 5. Rolling Risk Metrics

In [6]:
sharpe_results = []
for t in df["ticker"].unique():
    sub     = df[df["ticker"] == t].copy().sort_values("Date")
    rm      = sub["daily_return"].rolling(90).mean()
    rs      = sub["daily_return"].rolling(90).std()
    sub["sharpe_90d"] = (rm / rs) * np.sqrt(365)
    sharpe_results.append(sub)
df = pd.concat(sharpe_results, ignore_index=True)

summary = (
    df.groupby("ticker")
      .agg(
          trading_days      = ("Date", "count"),
          avg_daily_ret_pct = ("daily_return", lambda x: round(x.mean() * 100, 4)),
          ann_volatility    = ("vol_30d", "mean"),
          avg_rsi           = ("rsi", "mean"),
          pct_bull_days     = ("regime", lambda x: round((x == "Bull").mean() * 100, 1)),
          pct_bear_days     = ("regime", lambda x: round((x == "Bear").mean() * 100, 1)),
          vol_spikes_total  = ("vol_spike", "sum"),
      )
)
summary


,trading_days,avg_daily_ret_pct,ann_volatility,avg_rsi,pct_bull_days,pct_bear_days,vol_spikes_total
ticker,,,,,,,
BNB,1432,0.3298,0.8745,52.8131,39.9000,32.9000,73
BTC,1432,0.3246,0.7896,53.4309,52.9000,47.0000,63
ETH,1432,0.0401,0.9303,49.0167,30.9000,29.7000,66
SOL,1432,0.3442,1.2588,51.4396,30.5000,29.3000,64


## 6. Charts

### 6.1 BTC Technical Analysis (Candlestick + BB + RSI + MACD)

In [7]:
btc = df[df["ticker"] == "BTC"].sort_values("Date")

fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True,
    row_heights=[0.55, 0.25, 0.20],
    subplot_titles=["BTC/USD — Price & Bollinger Bands", "RSI (14)", "MACD"]
)

fig.add_trace(go.Candlestick(
    x=btc["Date"], open=btc["Open"], high=btc["High"],
    low=btc["Low"], close=btc["Close"], name="BTC",
    increasing_line_color="#26a69a", decreasing_line_color="#ef5350"
), row=1, col=1)

for col, color, name in [
    ("bb_upper", "rgba(255,165,0,0.8)", "BB Upper"),
    ("sma_20",   "rgba(255,255,255,0.6)", "SMA 20"),
    ("bb_lower", "rgba(255,165,0,0.8)", "BB Lower"),
]:
    fig.add_trace(go.Scatter(
        x=btc["Date"], y=btc[col],
        line=dict(color=color, width=1), name=name
    ), row=1, col=1)

fig.add_trace(go.Scatter(
    x=btc["Date"], y=btc["rsi"],
    line=dict(color="#7b68ee", width=1.5), name="RSI"
), row=2, col=1)
fig.add_hline(y=70, line_dash="dash", line_color="rgba(239,83,80,0.5)", row=2, col=1)
fig.add_hline(y=30, line_dash="dash", line_color="rgba(38,166,154,0.5)", row=2, col=1)

bar_colors = ["#26a69a" if v >= 0 else "#ef5350" for v in btc["macd_hist"]]
fig.add_trace(go.Bar(
    x=btc["Date"], y=btc["macd_hist"],
    marker_color=bar_colors, name="MACD Hist", opacity=0.7
), row=3, col=1)
fig.add_trace(go.Scatter(
    x=btc["Date"], y=btc["macd"],
    line=dict(color="#2196f3", width=1.2), name="MACD"
), row=3, col=1)
fig.add_trace(go.Scatter(
    x=btc["Date"], y=btc["macd_signal"],
    line=dict(color="#ff9800", width=1.2), name="Signal"
), row=3, col=1)

fig.update_layout(
    height=820, template="plotly_dark",
    paper_bgcolor="#0d1117", plot_bgcolor="#0d1117",
    xaxis_rangeslider_visible=False,
    title_text="BTC/USD Technical Analysis Dashboard", title_font_size=16
)
fig.show()


### 6.2 Volatility Heatmap

In [8]:
btc_m = (
    btc.set_index("Date")["vol_30d"]
       .resample("ME").mean()
       .reset_index()
)
btc_m["year"]  = btc_m["Date"].dt.year
btc_m["month"] = btc_m["Date"].dt.month_name().str[:3]

month_order = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
pivot = btc_m.pivot(index="year", columns="month", values="vol_30d")
pivot = pivot.reindex(columns=[m for m in month_order if m in pivot.columns])

fig2 = go.Figure(go.Heatmap(
    z=pivot.values,
    x=pivot.columns.tolist(),
    y=pivot.index.tolist(),
    colorscale="RdYlGn_r",
    colorbar=dict(title="Ann. Vol"),
    text=np.round(pivot.values, 2),
    texttemplate="%{text}",
))
fig2.update_layout(
    title="BTC Annualised Volatility Heatmap (30-Day Rolling)",
    template="plotly_dark", paper_bgcolor="#0d1117",
    plot_bgcolor="#0d1117", height=350
)
fig2.show()


### 6.3 Cross-Asset Correlation

In [9]:
returns_wide = df.pivot_table(index="Date", columns="ticker", values="log_return")
corr = returns_wide.corr()

fig3 = go.Figure(go.Heatmap(
    z=corr.values,
    x=corr.columns.tolist(),
    y=corr.index.tolist(),
    colorscale="RdBu", zmid=0,
    text=np.round(corr.values, 2),
    texttemplate="%{text}",
    colorbar=dict(title="Pearson r")
))
fig3.update_layout(
    title="Cross-Asset Log-Return Correlation Matrix (2021–2024)",
    template="plotly_dark", paper_bgcolor="#0d1117",
    plot_bgcolor="#0d1117", height=420
)
fig3.show()


### 6.4 Regime Distribution

In [10]:
regime_stats = (
    df.groupby(["ticker", "regime"])
      .agg(days=("Date","count"), avg_return=("daily_return","mean"), avg_vol=("vol_30d","mean"))
      .reset_index()
)

fig4 = px.bar(
    regime_stats, x="ticker", y="days", color="regime",
    barmode="group",
    color_discrete_map={"Bull":"#26a69a","Bear":"#ef5350","Sideways":"#ffa726"},
    title="Market Regime Distribution by Asset",
    template="plotly_dark", labels={"days":"Trading Days","ticker":"Asset"}
)
fig4.update_layout(paper_bgcolor="#0d1117", plot_bgcolor="#0d1117", height=420)
fig4.show()


### 6.5 Volume Spike Events

In [11]:
spike_counts = (
    df[df["vol_spike"] == True]
      .groupby(["ticker","year"])
      .size()
      .reset_index(name="spike_count")
)

fig5 = px.bar(
    spike_counts, x="year", y="spike_count", color="ticker",
    barmode="group",
    title="Volume Spike Events per Year (Volume > 2× 20-Day Average)",
    template="plotly_dark", color_discrete_map=colors_map
)
fig5.update_layout(paper_bgcolor="#0d1117", plot_bgcolor="#0d1117", height=420)
fig5.show()


### 6.6 90-Day Rolling Sharpe Ratio

In [12]:
fig6 = go.Figure()
for t in ["BTC","ETH","BNB","SOL"]:
    sub = df[df["ticker"] == t].sort_values("Date")
    fig6.add_trace(go.Scatter(
        x=sub["Date"], y=sub["sharpe_90d"],
        name=t, line=dict(color=colors_map[t], width=1.8)
    ))
fig6.add_hline(y=1, line_dash="dot", line_color="white", opacity=0.4)
fig6.add_hline(y=0, line_dash="solid", line_color="red", opacity=0.3)
fig6.update_layout(
    title="90-Day Rolling Sharpe Ratio — All Assets",
    template="plotly_dark", paper_bgcolor="#0d1117",
    plot_bgcolor="#0d1117", height=460,
    yaxis_title="Sharpe Ratio", xaxis_title="Date"
)
fig6.show()


### 6.7 Cumulative Returns

In [13]:
fig7 = go.Figure()
for t in ["BTC","ETH","BNB","SOL"]:
    sub = df[df["ticker"] == t].sort_values("Date").dropna(subset=["daily_return"])
    cum = (1 + sub["daily_return"]).cumprod()
    fig7.add_trace(go.Scatter(
        x=sub["Date"], y=cum,
        name=t, line=dict(color=colors_map[t], width=2)
    ))
fig7.update_layout(
    title="Cumulative Returns (Base = 1.0, Jan 2021)",
    template="plotly_dark", paper_bgcolor="#0d1117",
    plot_bgcolor="#0d1117", height=460, hovermode="x unified"
)
fig7.show()


### 6.8 Return Distribution

In [14]:
fig8 = go.Figure()
for t in ["BTC","ETH","BNB","SOL"]:
    sub = df[df["ticker"] == t].dropna(subset=["daily_return"])
    fig8.add_trace(go.Violin(
        x=sub["daily_return"] * 100, name=t,
        line_color=colors_map[t], fillcolor=colors_map[t],
        opacity=0.6, box_visible=True, meanline_visible=True, orientation="h"
    ))
fig8.update_layout(
    title="Daily Return Distribution by Asset (%)",
    template="plotly_dark", paper_bgcolor="#0d1117",
    plot_bgcolor="#0d1117", height=480, xaxis_title="Daily Return (%)"
)
fig8.show()


## 7. Business Insights

In [15]:
regime_pnl = (
    df.groupby(["ticker","regime"])["daily_return"]
      .agg(mean="mean", count="count", std="std")
      .reset_index()
)
regime_pnl["annualised_return_pct"] = (regime_pnl["mean"] * 365 * 100).round(1)

spike_df = df.copy()
spike_df["next_3d_return"] = (
    spike_df.groupby("ticker")["Close"]
             .transform(lambda x: x.shift(-3) / x - 1) * 100
)
spike_effect = (
    spike_df.groupby(["ticker","vol_spike"])["next_3d_return"]
             .mean().unstack()
             .rename(columns={False:"normal_days", True:"post_spike"})
             .round(2)
)

print("=" * 55)
print("INSIGHT 1 — Regime-Conditional Returns")
print("=" * 55)
print(regime_pnl[["ticker","regime","annualised_return_pct","count"]].to_string(index=False))

print()
print("=" * 55)
print("INSIGHT 2 — 3-Day Forward Return After Volume Spike")
print("=" * 55)
print(spike_effect.to_string())

print()
print("=" * 55)
print("INSIGHT 3 — Portfolio Risk Summary")
print("=" * 55)
print(summary.to_string())


INSIGHT 1 — Regime-Conditional Returns
ticker   regime  annualised_return_pct  count
   BNB     Bear              -566.3000    471
   BNB     Bull               837.9000    571
   BNB Sideways              -100.9000    390
   BTC     Bear              -471.8000    673
   BTC     Bull               643.3000    758
   BTC Sideways              -433.6000      1
   ETH     Bear              -795.3000    425
   ETH     Bull             1,138.5000    443
   ETH Sideways              -257.8000    564
   SOL     Bear            -1,210.1000    420
   SOL     Bull             1,448.1000    437
   SOL Sideways                96.2000    575

INSIGHT 2 — 3-Day Forward Return After Volume Spike
vol_spike  normal_days  post_spike
ticker                            
BNB             1.0000      1.0200
BTC             0.9900      0.6600
ETH             0.1200      0.4600
SOL             0.9300      2.2700

INSIGHT 3 — Portfolio Risk Summary
        trading_days  avg_daily_ret_pct  ann_volatility  avg_rsi